# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook builds the Lane 2 feature vector and then tries to break it. The prediction moment is the point immediately before the decline outcome is measured. Anything that defines the label, comes from the future window, identifies a client/content item, or encodes a product decision is kept out of the production feature vector.

The checks below are deliberately simple and reproducible: build the vector, document availability/missingness, run automated leakage traps, and record the excluded fields.


## 1. Build the feature vector

The production vector uses only fields available before the prediction outcome. Numeric features are coerced to numeric and median-imputed, with a missingness indicator where missingness itself is informative. Categorical fields are filled with `unknown` and one-hot encoded.

`trend_pct`, `trend_direction`, and `is_declining_label` are explicitly excluded because the label is derived from them.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

# Find the repository root so this runs from VS Code/Jupyter as well as from the repo root.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    ROOT = ROOT.parent

RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {RAW_PATH}. Run this notebook from inside the FLrank1 repository."
    )

df = pd.read_csv(RAW_PATH)

# Lane 2 target: declining traffic. It is used only as y/evaluation, never as X.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

# These are the features selected for the production vector.
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
]

CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

# Do not silently assume every release has every column.
missing_columns = [
    c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES
    if c not in df.columns
]
if missing_columns:
    raise KeyError(f"Required feature columns are missing: {missing_columns}")

# Numeric engineering.
X_num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").copy()

# Keep explicit missingness indicators. This avoids treating "missing" as an ordinary zero.
missing_flags = {}
for col in NUMERIC_FEATURES:
    missing_flags[f"{col}__missing"] = X_num[col].isna().astype("int8")
X_missing = pd.DataFrame(missing_flags, index=df.index)

# Median fill is learned from this analysis slice only; no label/future value is used.
X_num = X_num.fillna(X_num.median(numeric_only=True))

# Categorical handling.
X_cat = df[CATEGORICAL_FEATURES].astype("string").fillna("unknown")
X_cat = pd.get_dummies(X_cat, prefix=CATEGORICAL_FEATURES, dtype="int8")

X = pd.concat([X_num, X_missing, X_cat], axis=1)
y = df["is_declining_label"].copy()

print(f"Rows: {len(df):,}")
print(f"Raw columns: {df.shape[1]:,}")
print(f"Production numeric features: {len(NUMERIC_FEATURES)}")
print(f"Production categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"Final encoded feature columns: {X.shape[1]:,}")
print(f"Declining base rate: {y.mean():.1%}")
print(f"Feature matrix contains NaN: {X.isna().any().any()}")
print(f"Feature matrix contains inf: {np.isinf(X.select_dtypes(include=np.number)).any().any()}")


## 2. Feature notes (meaning, missing, categorical, available-when?)

The feature list is intentionally restricted to pre-outcome information. The 90-day totals, content metadata, freshness, position and derived rates describe the state known at the prediction point. Missing numeric values are median-filled and also get a `__missing` flag; categorical missing values become `unknown`.

The label source is not included in this table because it is deliberately not a feature.


In [ ]:
feature_notes = pd.DataFrame([
    ["search_volume", "Keyword demand estimate", "Median + missing flag", "Pre-outcome"],
    ["competition", "Keyword competition score", "Median + missing flag", "Pre-outcome"],
    ["cpc", "Keyword CPC estimate", "Median + missing flag", "Pre-outcome"],
    ["word_count", "Content length", "Median + missing flag", "Pre-outcome"],
    ["char_count", "Content character count", "Median + missing flag", "Pre-outcome"],
    ["impressions_90d", "Search visibility over trailing 90 days", "Median + missing flag", "Pre-outcome"],
    ["clicks_90d", "Search clicks over trailing 90 days", "Median + missing flag", "Pre-outcome"],
    ["pageviews_90d", "GA4 pageviews over trailing 90 days", "Median + missing flag", "Pre-outcome"],
    ["sessions_90d", "GA4 sessions over trailing 90 days", "Median + missing flag", "Pre-outcome"],
    ["users_90d", "GA4 users over trailing 90 days", "Median + missing flag", "Pre-outcome"],
    ["engaged_sessions_90d", "Engaged GA4 sessions over trailing 90 days", "Median + missing flag", "Pre-outcome"],
    ["ai_sessions_90d", "AI-referred sessions over trailing 90 days", "Median + missing flag", "Pre-outcome"],
    ["scroll_events_90d", "Scroll events over trailing 90 days", "Median + missing flag", "Pre-outcome"],
    ["days_with_impressions", "Days with search impressions", "Median + missing flag", "Pre-outcome"],
    ["days_with_sessions", "Days with sessions", "Median + missing flag", "Pre-outcome"],
    ["content_age_days", "Age of content", "Median + missing flag", "Pre-outcome"],
    ["days_since_last_update", "Freshness / time since update", "Median + missing flag", "Pre-outcome"],
    ["ctr", "Clicks divided by impressions", "Median + missing flag", "Pre-outcome"],
    ["avg_position", "Average GSC position", "Median + missing flag", "Pre-outcome; 0 is not treated as a real rank"],
    ["engagement_rate", "Engaged sessions / sessions", "Median + missing flag", "Pre-outcome"],
    ["scroll_rate", "Scroll events / pageviews", "Median + missing flag", "Pre-outcome"],
    ["ai_traffic_pct", "AI sessions / sessions", "Median + missing flag", "Pre-outcome"],
    ["competition_level", "Keyword competition bucket", "unknown + one-hot", "Pre-outcome"],
    ["content_type", "Content type", "unknown + one-hot", "Pre-outcome"],
    ["main_intent", "Search intent", "unknown + one-hot", "Pre-outcome"],
    ["age_tier", "Content age bucket", "unknown + one-hot", "Pre-outcome"],
    ["freshness_tier", "Update recency bucket", "unknown + one-hot", "Pre-outcome"],
    ["word_count_tier", "Word-count bucket", "unknown + one-hot", "Pre-outcome"],
    ["impression_tier", "Impression-volume bucket", "unknown + one-hot", "Pre-outcome"],
    ["position_tier", "Position bucket", "unknown + one-hot", "Pre-outcome"],
], columns=["feature", "meaning", "missing/categorical handling", "availability"])

display(feature_notes)

print("Numeric missingness before fill:")
display(
    df[NUMERIC_FEATURES].isna().mean()
      .sort_values(ascending=False)
      .head(10)
      .rename("missing_rate")
      .to_frame()
)

print("Categorical missingness:")
display(
    df[CATEGORICAL_FEATURES].isna().mean()
      .sort_values(ascending=False)
      .rename("missing_rate")
      .to_frame()
)


## 3. The leakage hunt

Three failure modes are tested explicitly:

1. **Label-derived leakage:** `trend_pct`, `trend_direction`, and `is_declining_label` must not appear in the production vector.
2. **Future-window leakage:** fields whose names describe the outcome/trend window must not appear in the production vector. The current feature set contains no `*_last_30d`, `*_prev_30d`, or trend fields.
3. **Identity/product leakage:** client/content IDs and product/decision fields are not allowed to become model features.

The last check is deliberately strict: it scans both the raw selected names and the final one-hot encoded feature names.


In [ ]:
# --- Static leakage tests ---

production_raw_features = set(NUMERIC_FEATURES + CATEGORICAL_FEATURES)
label_derived = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
}

future_window_patterns = (
    "last_30d",
    "prev_30d",
    "next_",
    "future",
    "forward",
)

identity_or_product_patterns = (
    "client_id",
    "content_id",
    "provider_used",
    "model_used",
    "product_flag",
    "flag_",
)

label_hits = sorted(production_raw_features & label_derived)
future_hits = sorted(
    c for c in production_raw_features
    if any(p in c.lower() for p in future_window_patterns)
)
identity_product_hits = sorted(
    c for c in production_raw_features
    if any(p in c.lower() for p in identity_or_product_patterns)
)

print("=== STATIC LEAKAGE TEST ===")
print("Label-derived fields found:", label_hits)
print("Future-window fields found:", future_hits)
print("Identity/product fields found:", identity_product_hits)

assert not label_hits, f"Label leakage found: {label_hits}"
assert not future_hits, f"Future-window leakage found: {future_hits}"
assert not identity_product_hits, f"Identity/product leakage found: {identity_product_hits}"

print("Static leakage check: PASS")

# The final encoded feature names should also contain none of the forbidden raw names.
encoded_forbidden = [
    c for c in X.columns
    if c in label_derived
    or any(p in c.lower() for p in future_window_patterns)
    or any(p in c.lower() for p in identity_or_product_patterns)
]
print("Forbidden patterns in encoded vector:", encoded_forbidden)
assert not encoded_forbidden


### Deliberate leak-trap experiment

A static test is necessary but not sufficient. To demonstrate the practical consequence, the next cell deliberately injects the true label as a fake feature, measures the score, removes the fake feature, and measures the honest score again.

This experiment is an audit only. `DELIBERATE_LABEL_LEAK` is never part of `X`.


In [ ]:
# --- Numeric leak-trap demonstration ---
# The experiment is intentionally artificial: it shows why the static guard matters.

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Keep this demonstration small enough to run quickly while remaining reproducible.
audit_cols = NUMERIC_FEATURES[:]
X_audit = df[audit_cols].apply(pd.to_numeric, errors="coerce").copy()
X_audit = X_audit.fillna(X_audit.median(numeric_only=True))
y_audit = y.astype(int)

Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_audit, y_audit, test_size=0.25, random_state=42, stratify=y_audit
)

def audit_auc(X_train, X_test, y_train, y_test):
    model = RandomForestClassifier(
        n_estimators=80,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

honest_auc = audit_auc(Xa_train, Xa_test, ya_train, ya_test)

# Deliberately leak the answer key.
X_leaky = X_audit.copy()
X_leaky["DELIBERATE_LABEL_LEAK"] = y_audit.to_numpy()

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky, y_audit, test_size=0.25, random_state=42, stratify=y_audit
)
leaky_auc = audit_auc(Xl_train, Xl_test, yl_train, yl_test)

# Remove the fake column and rerun.
X_clean = X_leaky.drop(columns=["DELIBERATE_LABEL_LEAK"])
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clean, y_audit, test_size=0.25, random_state=42, stratify=y_audit
)
clean_auc = audit_auc(Xc_train, Xc_test, yc_train, yc_test)

print("=== LEAK-TRAP EXPERIMENT ===")
print(f"Honest ROC-AUC:       {honest_auc:.4f}")
print(f"With deliberate leak: {leaky_auc:.4f}")
print(f"After removing leak:  {clean_auc:.4f}")
print(f"Leak uplift:          {leaky_auc - honest_auc:+.4f}")

assert leaky_auc > honest_auc, "The deliberate leak did not increase the score."
assert abs(clean_auc - honest_auc) < 1e-12, "Removing the leak did not restore the honest score."
assert "DELIBERATE_LABEL_LEAK" not in X.columns

print("Leak-trap check: PASS")


## 4. What I excluded and why

These fields are intentionally refused as model inputs:

- `trend_pct` — it directly defines the declining label.
- `trend_direction` — it is the categorical source of the declining label.
- `is_declining_label` — it is the target itself.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` — recent trend-window fields are too close to the outcome definition for this task.
- `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` — these belong to the trend-label construction and are excluded from this feature vector rather than relying on accidental window alignment.
- `client_id`, `content_id` — identifiers for grouping/joining, not predictive content attributes.
- `provider_used`, `model_used` — generation metadata; they can encode implementation/product information rather than the content state an editor is meant to act on.


In [ ]:
excluded_fields = pd.DataFrame([
    ["trend_pct", "Direct source of the decline label; label-derived leakage."],
    ["trend_direction", "Defines the decline label; label-derived leakage."],
    ["is_declining_label", "Target variable; using it would reveal the answer."],
    ["impressions_last_30d", "Part of the recent trend/outcome window."],
    ["clicks_last_30d", "Part of the recent trend/outcome window."],
    ["sessions_last_30d", "Part of the recent trend/outcome window."],
    ["impressions_prev_30d", "Part of the trend-label construction."],
    ["clicks_prev_30d", "Part of the trend-label construction."],
    ["sessions_prev_30d", "Part of the trend-label construction."],
    ["client_id", "Pseudonymous identity; reserved for grouping/splitting."],
    ["content_id", "Pseudonymous identity; reserved for joins/grouping."],
    ["provider_used", "Generation metadata; excluded from the production feature vector."],
    ["model_used", "Generation metadata; excluded from the production feature vector."],
], columns=["field", "reason"])

display(excluded_fields)

print("Excluded fields present in raw data:")
print([c for c in excluded_fields["field"] if c in df.columns])


## Self-check

The notebook should finish with all checks passing.

- [x] Feature vector is built from explicit pre-outcome fields.
- [x] Numeric and categorical missingness is handled explicitly.
- [x] Label-derived fields are excluded.
- [x] Future/trend-window fields are excluded.
- [x] IDs and product/generation metadata are excluded.
- [x] Static leakage test is shown.
- [x] Numeric leak-trap experiment demonstrates the before/after effect.
- [x] No client names, private queries, or secrets are used.
- [ ] Run the complete notebook top-to-bottom locally and commit the executed notebook.
